# NVIDIA GPU Operator — MIG Configuration

A practical reference for managing **Multi-Instance GPU (MIG)** through the
**NVIDIA GPU Operator** on Kubernetes. The GPU Operator automates the full
lifecycle of the software a GPU node needs — the NVIDIA driver, the
container toolkit, the Kubernetes **device plugin**, **GPU Feature Discovery
(GFD)**, **DCGM Exporter**, the **Node Feature Discovery (NFD)** add-on, and
the **MIG Manager** — so that partitioning an A100/A30/H100/H200/B200 into
isolated GPU instances becomes a single label on a node rather than a series
of manual `nvidia-smi mig` commands run by hand on every host.

This notebook focuses on the **MIG-specific** parts of the GPU Operator: the
`mig.strategy` setting, the `nvidia.com/mig.config` node label, the
`mig-parted` ConfigMap that defines the partition layouts, and how MIG devices
are surfaced to pods as schedulable resources.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

The **NVIDIA GPU Operator** is a Kubernetes Operator (built on the
Operator Framework) that installs and manages all of the NVIDIA software
components required to run GPU workloads in a cluster. **MIG configuration**
is the subset of that automation responsible for partitioning MIG-capable
GPUs into smaller, hardware-isolated **GPU Instances (GIs)** and
**Compute Instances (CIs)**, and for advertising those partitions to the
Kubernetes scheduler.

Without the Operator you would: install the driver, enable MIG mode with
`nvidia-smi -mig 1`, reset the GPU, create GIs/CIs with `nvidia-smi mig
create`, and configure the device plugin — on every node, repeating the work
after every reboot or driver upgrade. The GPU Operator collapses all of that
into declarative state: you label a node with the MIG layout you want and the
**MIG Manager** reconciles the hardware to match.

### Why use it?

- **Declarative MIG**: the desired partition layout is a node label
  (`nvidia.com/mig.config=all-1g.10gb`), not an imperative script. The
  Operator drives the GPU to that state and keeps it there across reboots.
- **No manual driver/toolkit install**: the Operator ships and version-pins
  the driver, container toolkit, and device plugin, so MIG works on a freshly
  provisioned node with no host-level setup.
- **Scheduler integration**: MIG slices are exposed as standard extended
  resources (`nvidia.com/gpu` or `nvidia.com/mig-1g.10gb`), so pods request
  them with ordinary resource limits.
- **Fleet consistency**: one `mig-parted` ConfigMap describes the valid
  layouts for the whole cluster; per-node labels select among them.

### When to use it?

- You run MIG-capable GPUs (A100, A30, H100, H200, GH200, B200) on Kubernetes
  and want many small, isolated workloads (inference servers, notebooks,
  CI jobs) to share each physical GPU safely.
- You need MIG geometry to survive node reboots and driver upgrades without
  human intervention.
- You manage more than a handful of GPU nodes and hand-running `nvidia-smi
  mig create` does not scale.
- You want quality-of-service isolation (one tenant cannot starve another's
  SMs or memory bandwidth) that time-slicing and MPS cannot provide.

## Key Features

### Core Capabilities

| Feature | Description | Benefit |
|---------|-------------|---------|
| `mig.strategy` (single / mixed) | Controls how MIG devices are advertised: `single` exposes them as homogeneous `nvidia.com/gpu`; `mixed` exposes each profile as `nvidia.com/mig-<profile>` | Pick a model that matches whether nodes hold uniform or heterogeneous slices |
| MIG Manager (`mig-parted`) | DaemonSet that watches the `nvidia.com/mig.config` label and reconciles GPUs to the named layout | Declarative, self-healing MIG geometry |
| `mig-parted` ConfigMap | Named, reusable MIG layouts (`all-1g.10gb`, `all-balanced`, custom) selectable per node | One source of truth for valid partitions cluster-wide |
| GPU Feature Discovery | Labels nodes with MIG profiles, memory, and product so the scheduler can place pods correctly | Topology-aware scheduling without manual labels |
| Device plugin | Registers MIG slices as Kubernetes extended resources | Pods request slices with normal `resources.limits` |
| WithReboot option | MIG mode toggling can require a GPU/node reset; the Manager can drain and reboot when needed | Safe transition into/out of MIG mode on bare metal |

## Architecture Overview

The GPU Operator deploys a stack of components into the `gpu-operator`
namespace. For MIG, the relevant flow is: a node label declares the desired
layout, the MIG Manager reconciles the hardware, and GFD + the device plugin
publish the resulting slices to the scheduler.

```
                         ┌──────────────────────────────────────────┐
                         │             Control Plane                 │
                         │   gpu-operator (ClusterPolicy CR)         │
                         └──────────────────────────────────────────┘
                                            │ reconciles
                                            ▼
  GPU node (label: nvidia.com/mig.config=all-1g.10gb)
  ┌──────────────────────────────────────────────────────────────────┐
  │  NFD ──▶ labels node (PCI IDs, MIG-capable)                       │
  │  Driver DaemonSet ──▶ kernel modules + nvidia-smi                 │
  │  Container Toolkit ──▶ nvidia container runtime                   │
  │                                                                    │
  │  MIG Manager (mig-parted) ── watches mig.config label ──┐          │
  │       │ enables MIG mode, creates GIs/CIs               │          │
  │       ▼                                                 │          │
  │  ┌──────────── Physical A100/H100 ────────────────┐     │          │
  │  │  GI 0 (1g.10gb)  GI 1 (1g.10gb)  ... GI 6       │     │          │
  │  └─────────────────────────────────────────────────┘    │          │
  │       │                                                 │          │
  │  GPU Feature Discovery ──▶ node labels (mig profiles) ◀─┘          │
  │  Device Plugin ──▶ advertises nvidia.com/mig-1g.10gb: 7            │
  └──────────────────────────────────────────────────────────────────┘
                                            │
                                            ▼
                         kube-scheduler places pods onto slices
```

### Components

1. **ClusterPolicy CR**: the single custom resource that configures the whole
   Operator, including `spec.mig.strategy` and the `migManager` block.
2. **MIG Manager DaemonSet** (`nvidia-mig-manager`): runs `mig-parted` on each
   MIG-capable node, watches the `nvidia.com/mig.config` label, and applies the
   matching layout from the ConfigMap.
3. **`default-mig-parted-config` ConfigMap**: maps human-readable layout names
   to concrete MIG geometries; you can extend it with custom layouts.
4. **GPU Feature Discovery + NFD**: discover hardware and publish per-node MIG
   labels (`nvidia.com/mig.config.state`, `nvidia.com/gpu.product`, etc.).
5. **Device Plugin**: turns each MIG slice into a schedulable extended resource.

## Installation

### Prerequisites

- A Kubernetes cluster (v1.21+) with at least one MIG-capable GPU node
  (A100, A30, H100, H200, GH200, or B200).
- `kubectl` configured against the cluster and `helm` v3 installed.
- Nodes **without** a pre-installed NVIDIA driver if you let the Operator manage
  the driver (the default), or with a matching pre-installed driver if you set
  `driver.enabled=false`.
- Container runtime that supports the NVIDIA runtime (containerd or CRI-O).

### Installation Steps

Install the GPU Operator with the MIG Manager enabled via Helm. The MIG
strategy is chosen at install time and can be changed later by patching the
ClusterPolicy.

**Note**: Run the shell commands below from a machine with cluster access; the
Python cell at the end of this section is a small helper for generating the
node-labeling command.

In [ ]:
# Add the NVIDIA Helm repo and install the GPU Operator with MIG enabled.
# Run these in a shell with kubectl/helm access (shown here as reference).

install_commands = r'''
helm repo add nvidia https://helm.ngc.nvidia.com/nvidia
helm repo update

# mig.strategy can be "single" (homogeneous slices) or "mixed" (heterogeneous).
helm install --wait gpu-operator nvidia/gpu-operator \
  --namespace gpu-operator --create-namespace \
  --set mig.strategy=single \
  --set migManager.enabled=true

# Verify the operator and MIG manager are running:
kubectl get pods -n gpu-operator
kubectl get pods -n gpu-operator -l app=nvidia-mig-manager
'''
print(install_commands)

## Basic Usage

### Quick Start: partition a node and run a pod

Once the Operator is running, MIG is driven by a single node label. Set
`nvidia.com/mig.config` to a layout name from the `mig-parted` ConfigMap; the
MIG Manager enables MIG mode (resetting the GPU if needed) and creates the
instances. Watch `nvidia.com/mig.config.state` go from `pending` to `success`.

In [ ]:
# Reference workflow: label a node, watch reconciliation, request a slice.

workflow = r'''
# 1) Pick a layout and apply it to the node (here: all 1g.10gb slices on an A100).
kubectl label node <gpu-node> nvidia.com/mig.config=all-1g.10gb --overwrite

# 2) Watch the MIG Manager reconcile. State flows: pending -> success (or failed).
kubectl get node <gpu-node> -o jsonpath='{.metadata.labels.nvidia\.com/mig\.config\.state}{"\n"}'
kubectl logs -n gpu-operator -l app=nvidia-mig-manager --tail=50

# 3) Confirm the slices are advertised as schedulable resources.
kubectl get node <gpu-node> -o jsonpath='{.status.allocatable}{"\n"}' | tr ',' '\n' | grep nvidia
'''
print(workflow)

In [ ]:
# A pod that requests a single MIG slice. With mig.strategy=single, MIG slices
# are advertised as nvidia.com/gpu, so the request is identical to a whole GPU.

pod_single = r'''
apiVersion: v1
kind: Pod
metadata:
  name: mig-single-demo
spec:
  restartPolicy: Never
  containers:
  - name: cuda
    image: nvidia/cuda:12.4.1-base-ubuntu22.04
    command: ["nvidia-smi", "-L"]
    resources:
      limits:
        nvidia.com/gpu: 1   # one 1g.10gb slice under mig.strategy=single
'''

# With mig.strategy=mixed, request a specific profile by name instead:
pod_mixed_limits = '''
    resources:
      limits:
        nvidia.com/mig-1g.10gb: 1
'''
print(pod_single)
print("# mixed-strategy resource request:")
print(pod_mixed_limits)

## Advanced Features

### Custom MIG layouts via the `mig-parted` ConfigMap

The Operator ships a `default-mig-parted-config` ConfigMap with common layouts
(`all-disabled`, `all-1g.10gb`, `all-2g.20gb`, `all-3g.40gb`, `all-balanced`,
…). You can supply your own ConfigMap to define mixed geometries — e.g. a few
large slices for training alongside many small slices for inference on the
same physical GPU. Each named layout becomes a valid value for the
`nvidia.com/mig.config` node label.

In [ ]:
# Custom mig-parted ConfigMap with a heterogeneous layout for an A100-40GB.
# 'custom-mixed' carves the GPU into one 3g.20gb + two 1g.5gb slices.

custom_config = r'''
apiVersion: v1
kind: ConfigMap
metadata:
  name: custom-mig-parted-config
  namespace: gpu-operator
data:
  config.yaml: |
    version: v1
    mig-configs:
      all-disabled:
        - devices: all
          mig-enabled: false
      custom-mixed:
        - devices: all
          mig-enabled: true
          mig-devices:
            "3g.20gb": 1
            "1g.5gb": 2
'''

# Point the operator at the custom ConfigMap, then select the layout per node:
wire_up = r'''
kubectl apply -f custom-mig-parted-config.yaml
kubectl patch clusterpolicy/cluster-policy --type merge \
  -p '{"spec":{"migManager":{"config":{"name":"custom-mig-parted-config"}}}}'
kubectl label node <gpu-node> nvidia.com/mig.config=custom-mixed --overwrite
'''
print(custom_config)
print(wire_up)

### `single` vs `mixed` strategy

- **`single`**: every MIG device on a node is the *same* profile. The device
  plugin advertises them all as `nvidia.com/gpu`, so application manifests are
  unchanged from whole-GPU scheduling. Simplest model; best when each node hosts
  a uniform slice size.
- **`mixed`**: a node may host *different* profiles. Each profile is advertised
  as a distinct resource (`nvidia.com/mig-1g.10gb`, `nvidia.com/mig-3g.40gb`,
  …). Pods must request the exact profile they need. Required for the custom
  heterogeneous layout above.

Switch strategies by patching the ClusterPolicy; the change is cluster-wide.

In [ ]:
patch_strategy = r'''
kubectl patch clusterpolicy/cluster-policy --type merge \
  -p '{"spec":{"mig":{"strategy":"mixed"}}}'
'''
print(patch_strategy)

## Use Cases

### Multi-tenant inference serving

- **Context**: many small models or low-traffic endpoints, each needing a few
  GB and a slice of compute, must share expensive A100/H100 hardware with hard
  isolation between tenants.
- **Implementation**: label nodes `all-1g.10gb` (single strategy) so each A100
  exposes seven `nvidia.com/gpu` slices; run one Triton/vLLM replica per slice.
- **Results**: up to 7× the number of isolated serving replicas per GPU, with
  no noisy-neighbor interference on memory bandwidth or SMs.

### Shared developer / notebook clusters

- **Context**: data scientists need on-demand GPU for interactive work but
  rarely saturate a full device.
- **Implementation**: a `mixed` layout offering a range of profiles
  (`1g.10gb`, `2g.20gb`, `3g.40gb`); users request the size they need.
- **Results**: higher utilization and predictable per-user quotas without
  over-provisioning whole GPUs.

## Best Practices

1. **Drain before relabeling**: changing `nvidia.com/mig.config` destroys and
   recreates GPU instances. Cordon/drain GPU workloads first, or enable
   `WITH_REBOOT`/the Manager's draining behavior, so running pods are not
   killed mid-job.
2. **Keep one source of truth for layouts**: define all valid geometries in a
   single `mig-parted` ConfigMap and select per node via the label — don't run
   `nvidia-smi mig` by hand on Operator-managed nodes.
3. **Match strategy to node uniformity**: use `single` when every slice on a
   node is identical; reach for `mixed` only when you genuinely need
   heterogeneous profiles.
4. **Pin component versions**: set explicit driver/toolkit/operator versions in
   the Helm values so a node re-provision yields a reproducible stack.
5. **Confirm `mig.config.state=success`** in automation before scheduling onto
   a node; treat `failed` as a hard gate.
6. **Reserve GPU memory headroom**: MIG profiles include framebuffer overhead;
   size your model footprints below the advertised slice memory.

## Common Pitfalls

1. **Forgetting MIG mode needs a GPU reset**: enabling/disabling MIG mode can
   require draining the node and resetting the GPU. If the GPU is in use by a
   pod, reconciliation stalls in `pending`. Drain first or allow the Manager to
   reboot.
2. **Strategy/label mismatch**: requesting `nvidia.com/mig-1g.10gb` while the
   cluster is on `single` strategy (which only advertises `nvidia.com/gpu`)
   leaves pods unschedulable. Match the request to the active strategy.
3. **Unsupported profile for the GPU**: not every profile exists on every GPU
   (an A30 has fewer slices than an A100). An invalid layout reports `failed`.
4. **Editing GPUs out-of-band**: hand-running `nvidia-smi mig create` on a
   node the Operator manages causes the Manager to revert your changes on the
   next reconcile.
5. **Expecting MIG on consumer GPUs**: MIG is data-center-only; the label is a
   no-op on non-MIG hardware.

## Performance Optimization

### Choosing the right slice size

MIG gives **hard isolation**, but each slice is a fixed fraction of SMs and
memory — there is no bursting beyond the slice. Tuning is about right-sizing:

- **Profile granularity**: smaller profiles (`1g.10gb`) maximize replica count
  but cap per-replica throughput. Larger profiles (`3g.40gb`, `7g.80gb`) suit
  bigger models. Benchmark your model on each candidate profile.
- **Memory vs compute balance**: profiles scale memory and compute together in
  fixed steps; pick the smallest profile whose memory fits your model plus KV
  cache / activation headroom.
- **Avoid fragmentation**: a `mixed` layout can strand capacity (e.g. a leftover
  `1g` slice no workload requests). Prefer uniform `single` layouts unless
  heterogeneity is required.
- **NUMA / topology awareness**: keep GFD and NFD enabled so the scheduler
  places latency-sensitive pods on the right node.

In [ ]:
# Compute how many replicas of a given footprint fit per GPU for each profile.
# Slice memory values are the usable framebuffer per profile on an A100-80GB.

A100_80GB_PROFILES = {
    "1g.10gb": {"mem_gb": 10, "slices_per_gpu": 7},
    "2g.20gb": {"mem_gb": 20, "slices_per_gpu": 3},
    "3g.40gb": {"mem_gb": 40, "slices_per_gpu": 2},
    "7g.80gb": {"mem_gb": 80, "slices_per_gpu": 1},
}

def replicas_per_gpu(model_gb: float, headroom_gb: float = 1.5):
    """Pick the smallest profile that fits, and report replicas per GPU."""
    need = model_gb + headroom_gb
    fits = [(p, v) for p, v in A100_80GB_PROFILES.items() if v["mem_gb"] >= need]
    if not fits:
        return f"model needs {need:.1f} GB — too large for any single MIG slice"
    profile, v = min(fits, key=lambda kv: kv[1]["mem_gb"])
    return f"{model_gb} GB model -> use {profile}: {v['slices_per_gpu']} replicas/GPU"

for size in (3, 8, 18, 35):
    print(replicas_per_gpu(size))

## Production Deployment

### Helm values for a production install

Drive the Operator from a versioned `values.yaml` rather than `--set` flags so
the install is reproducible and reviewable. Enable the MIG Manager and pin the
strategy and the layout ConfigMap.

```yaml
# values.yaml for: helm install gpu-operator nvidia/gpu-operator -f values.yaml
driver:
  enabled: true
  version: "550.90.07"
mig:
  strategy: single
migManager:
  enabled: true
  env:
    - name: WITH_REBOOT       # allow node reboot when toggling MIG mode
      value: "true"
  config:
    name: custom-mig-parted-config
    default: all-disabled
gfd:
  enabled: true
dcgmExporter:
  enabled: true
```

### Pinning nodes to a layout at provisioning time

Apply the `nvidia.com/mig.config` label as part of node bootstrap (cloud-init,
Cluster API, or a `kubectl label` step in your provisioning pipeline) so a new
node self-partitions the moment the Operator schedules onto it.

```bash
kubectl label node <gpu-node> nvidia.com/mig.config=all-1g.10gb --overwrite
```

## Monitoring and Observability

### Key Metrics to Track

- **`nvidia.com/mig.config.state`** (node label): `success`, `pending`, or
  `failed` — the primary health signal for MIG reconciliation.
- **Per-slice utilization**: DCGM Exporter emits GPU and MIG-instance level
  metrics (`DCGM_FI_DEV_GPU_UTIL`, profile/GI tags) scraped by Prometheus.
- **Allocatable vs requested slices**: track `nvidia.com/gpu` (or
  `nvidia.com/mig-*`) allocatable vs requests to spot stranded capacity.
- **MIG Manager pod health**: the `nvidia-mig-manager` DaemonSet should be
  Ready on every MIG-capable node.

### Useful commands

```bash
# Reconciliation state across all GPU nodes:
kubectl get nodes -L nvidia.com/mig.config,nvidia.com/mig.config.state

# Inspect actual MIG geometry on a node:
kubectl exec -n gpu-operator <mig-manager-pod> -- nvidia-smi mig -lgi

# DCGM exporter metrics endpoint (scraped by Prometheus):
kubectl port-forward -n gpu-operator svc/nvidia-dcgm-exporter 9400:9400
```

## Troubleshooting

### Issue 1: `mig.config.state` stuck on `pending`

**Symptoms**: the node label never reaches `success`; MIG geometry doesn't
change.

**Cause**: toggling MIG mode requires the GPU to be idle; running pods (or a
GPU that needs a reset) block the transition.

**Solution**: cordon and drain the node, or set `WITH_REBOOT=true` on the MIG
Manager so it can reboot to apply the change; then re-check the state.

### Issue 2: `mig.config.state` reports `failed`

**Symptoms**: the label value is `failed` right after relabeling.

**Cause**: the requested layout is invalid for that GPU (unsupported profile or
a count the device can't satisfy), or the layout name isn't in the ConfigMap.

**Solution**: check `kubectl logs -n gpu-operator -l app=nvidia-mig-manager`,
confirm the layout name exists in the `mig-parted` ConfigMap, and verify the
profile is valid for the GPU model.

### Issue 3: pods stay `Pending` with insufficient GPU

**Symptoms**: a pod requesting a MIG slice never schedules.

**Cause**: strategy/label mismatch — requesting `nvidia.com/mig-1g.10gb` under
`single` strategy, or `nvidia.com/gpu` while no homogeneous slices exist.

**Solution**: align the resource request with the active `mig.strategy`; verify
advertised resources with `kubectl describe node <gpu-node>`.

## Comparison with Alternatives

### GPU Operator MIG vs other GPU-sharing approaches

| Aspect | GPU Operator MIG | Time-Slicing | MPS | Manual `nvidia-smi mig` |
|--------|------------------|--------------|-----|-------------------------|
| Isolation | Hardware (SMs, mem, BW) | None (context switch) | Soft (shared context) | Hardware |
| Memory protection | Yes, per slice | No | No | Yes |
| Hardware required | A100/A30/H100/H200/B200 | Any GPU | Volta+ | MIG-capable |
| Declarative on K8s | Yes (node label) | Yes (config) | Partial | No (imperative) |
| Survives reboot | Yes (reconciled) | Yes | Yes | No (manual redo) |
| Best for | Multi-tenant isolation | Bursty, trusted sharing | Cooperative MPI/many small kernels | One-off / non-K8s hosts |

### When to choose GPU Operator MIG

- You need **hardware isolation** between untrusted or quota-bound tenants.
- You run MIG-capable GPUs on **Kubernetes** and want partitioning to be
  declarative and self-healing.
- You manage a **fleet** and cannot afford per-node manual MIG setup.

Prefer **time-slicing** for bursty, trusted workloads on any GPU; prefer **MPS**
for cooperative many-small-kernel jobs that benefit from a shared context.

## Resources

### Official Documentation

- GPU Operator docs: https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/index.html
- MIG support in the GPU Operator: https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-operator-mig.html
- MIG User Guide: https://docs.nvidia.com/datacenter/tesla/mig-user-guide/

### Tooling and Source

- GPU Operator on GitHub: https://github.com/NVIDIA/gpu-operator
- `mig-parted` (MIG Partition Editor): https://github.com/NVIDIA/mig-parted
- Kubernetes device plugin: https://github.com/NVIDIA/k8s-device-plugin

### Related Components in this Stack

- MIG Manager — the DaemonSet that reconciles geometry (`mig-manager` notebook)
- Multi-Instance GPU — the underlying hardware feature (`multi-instance-gpus`)
- DCGM Exporter — per-slice metrics for Prometheus (`dcgm-exporter`)
- GPU Feature Discovery — node labeling (`gpu-feature-discovery`)
- Time-Slicing and MPS — alternative sharing models (`time-slicing`, `multi-process-service`)